In [ ]:
import pandas as pd
import spacy
import random
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

# 1. Carga do modelo spaCy
nlp = spacy.load('pt_core_news_sm')

# 2. Base de dados de Exemplo (DADOS ROTULADOS)
# Modelos supervisionados exigem dados com a resposta correta para parender padrões.
categorias = ['positivo', 'negativo', 'neutro']
frases_base = {
    'positivo': ['Essa campanha está incrível', 'Adorei o vídeo', 'Melhor propaganda do ano', 'Excelente iniciativa', 'Estou amando'],
    'negativo': ['Odiei o anúncio', 'Muito confuso e ruim', 'Que porcaria de serviço', 'Propaganda enganosa', 'Horrível'],
    'neutro': ['Achei mediano', 'Não fede nem cheira', 'Nada de especial', 'Apenas ok', 'Indiferente']
    
}

dados_expandidos = []
for i in range(500):
    cat = random.choice(categorias) # Garante o equilibrio entre as classes
    texto = random.choice(frases_base[cat])
    dados_expandidos.append([texto, cat])

df = pd.DataFrame(dados_expandidos, columns=['comentario', 'sentimento'])

# 3. Pré-processamento com spaCy (Tokenização e Normalização)
# A normalização converte as palavras para sua forma base (lemma) e remove ruídos.
def limpar_texto(texto):
    doc = nlp(texto.lower())
    # Retorna os lemmas ignornado pontuações
    return " ".join([token.lemma_ for token in doc if not token.is_punct and not token.is_stop and not token.is_space])

df['texto_limpo'] = df['comentario'].apply(limpar_texto)

# 4. Vetorização (Bag of Words)
# O CountVectorizer transforma os fragmentos de texto em vetores numéricos.
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['texto_limpo'])
y = df['sentimento']

# 5. Treinamento do Modelo Naive Bayes
# O algoritmo aprende a associar palavras-chave a categorias de sentimento.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=50)
modelo = MultinomialNB()
modelo.fit(X_train, y_train)

# 6. Aplicação em Novos Comentários (Simulação da Campanha)
novos_comentarios = [
    'Estou amando as novidades!',
    'Que porcaria de serviço.',
    'Achei mediano, nada de especial.'
]

# Tratar e vetorizar os novos dados
novos_limpos = [limpar_texto(c) for c in novos_comentarios]
X_novos = vectorizer.transform(novos_limpos)
predicoes = modelo.predict(X_novos)

# Exibição dos Resultados
print("\n--- RESULTADOS DA ANÁLISE BÁSICA DE SENTIMENTOS ---\n")
for comentario, sentimento in zip(novos_comentarios, predicoes):
    print(f"Comentário: {comentario} | Sentimento Predito: {sentimento}")

# Avaliação básica da precisão
acuracia = modelo.score(X_test, y_test)
print(f"\nAcurácia do modelo nos dados de teste: {acuracia * 100:.2f}%")

# Avaliação com métricas profissionais
print("\n--- MÉTRICAS DE DESEMPENHO PROFISSIONAIS ---")
print("\nRelatório de Classificação:\n")
print(classification_report(y_test, modelo.predict(X_test)))
acuracia = modelo.score(X_test, y_test)
print(f"Acurácia nos dados de teste: {acuracia * 100:.2f}%")


--- RESULTADOS DA ANÁLISE BÁSICA DE SENTIMENTOS ---

Comentário: Estou amando as novidades! | Sentimento Predito: positivo
Comentário: Que porcaria de serviço. | Sentimento Predito: negativo
Comentário: Achei mediano, nada de especial. | Sentimento Predito: neutro

Acurácia do modelo nos dados de teste: 100.00%

--- MÉTRICAS DE DESEMPENHO PROFISSIONAIS ---

Relatório de Classificação:

              precision    recall  f1-score   support

    negativo       1.00      1.00      1.00        36
      neutro       1.00      1.00      1.00        40
    positivo       1.00      1.00      1.00        49

    accuracy                           1.00       125
   macro avg       1.00      1.00      1.00       125
weighted avg       1.00      1.00      1.00       125

Acurácia nos dados de teste: 100.00%
